# Regularization, cell by cell

Goodfellow et al., [ch. 7](https://www.deeplearningbook.org/contents/regularization.html). Companion notes: `NOTES.md`.

**How to use this:** run top to bottom once so Fashion-MNIST is cached, then jump around. Every technique cell has a knob (`ALPHA`, `DROPOUT`, …) — change it, re-run **that cell**.

Data is not synthetic: sklearn `diabetes` / `breast_cancer`, plus official Fashion-MNIST (first fashion cell downloads ~30MB).

Default train budget is small so cells finish in a few seconds. Bump `EPOCHS` / `N_TRAIN` in the setup cell if you want the gaps to look more like `run.py`.

| # | section | knob |
|---|---|---|
| 2 | \(L_2\) geometry (Fig 7.1) | `ALPHA` |
| 3 | \(L_1\) vs \(L_2\) shrink | `ALPHA` |
| 5 | diabetes ridge / lasso | `RIDGE_ALPHA`, `LASSO_ALPHA` |
| 6 | cancer logreg | `C_L1`, `C_L2` |
| 8–14 | MLP regularizers | `WD`, `L1`, `DROPOUT`, `NOISE`, `EPS`, `PATIENCE` |
| 15–16 | CNN + aug | `MAX_SHIFT`, `N_COPIES` |
| 17 | sparse activations | `ACT_L1` |
| 18 | bagging | `N_BAG` |
| 19 | FGSM | `ADV_EPS` |

## 0. Setup

Run this first. It puts `regularization/` on `sys.path` whether you opened the notebook from the repo root or from inside the folder.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = "retina"

import os, sys
from pathlib import Path

HERE = Path.cwd().resolve()
if (HERE / "regularization" / "train.py").exists():
    HERE = HERE / "regularization"
elif not (HERE / "train.py").exists():
    raise FileNotFoundError(f"can't find train.py from {Path.cwd()}")
os.chdir(HERE)
sys.path.insert(0, str(HERE))
print("cwd:", HERE)

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge

from closed_form import l1_soft_threshold, l2_shrink_diag, l2_shrink_eigen, ridge_normal_equation
from data import (
    FASHION_LABELS,
    load_cancer_split,
    load_diabetes_split,
    load_fashion_split,
    shift_images,
)
from models import MLP, TinyCNN, fgsm, n_params, weight_l1
from train import (
    TrainConfig,
    bagged_proba,
    bootstrap_indices,
    metrics_for,
    train_classifier,
)

plt.rcParams.update({"figure.figsize": (6.2, 4.0), "figure.dpi": 110})

# --- knobs for the neural-net cells (re-run setup, then re-run those cells) ---
EPOCHS = 8
N_TRAIN = 800
N_VAL = 800
N_TEST = 1500
LR = 3e-3
HIDDEN = (256, 256)
SEED = 0

BOARD: dict[str, dict] = {}
MODELS: dict[str, torch.nn.Module] = {}


def show_board() -> None:
    if not BOARD:
        print("board empty — train something first")
        return
    print(f"{'name':18} {'train':>7} {'val':>7} {'test':>7} {'gap':>7}")
    for k, m in BOARD.items():
        print(f"{k:18} {m['train']:7.3f} {m['val']:7.3f} {m['test']:7.3f} {m['gap']:7.3f}")


def param_l2(model: torch.nn.Module) -> float:
    tot = 0.0
    for p in model.parameters():
        if p.ndim > 1:
            tot += float(p.detach().pow(2).sum())
    return tot ** 0.5


def plot_hist(hist, title: str = "") -> None:
    fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.3))
    ax[0].plot(hist.train_acc, label="train")
    ax[0].plot(hist.val_acc, label="val")
    ax[0].set_title("accuracy")
    ax[0].set_xlabel("epoch")
    ax[0].legend()
    ax[1].plot(hist.train_loss, label="train")
    ax[1].plot(hist.val_loss, label="val")
    if getattr(hist, "best_epoch", None) is not None:
        ax[1].axvline(hist.best_epoch, ls=":", color="k", lw=1)
    ax[1].set_title("loss")
    ax[1].set_xlabel("epoch")
    ax[1].legend()
    fig.suptitle(title, y=1.02)
    plt.show()


def fit_mlp(name: str, dropout: float = 0.0, **cfg) -> tuple:
    data = DATA
    torch.manual_seed(cfg.get("seed", SEED))
    model = MLP(data.n_features, data.n_classes, hidden=HIDDEN, dropout=dropout)
    tc = TrainConfig(
        epochs=cfg.get("epochs", EPOCHS),
        lr=cfg.get("lr", LR),
        batch_size=64,
        optimizer="adamw",
        weight_decay=cfg.get("weight_decay", 0.0),
        l1=cfg.get("l1", 0.0),
        activation_l1=cfg.get("activation_l1", 0.0),
        input_noise=cfg.get("input_noise", 0.0),
        label_smoothing=cfg.get("label_smoothing", 0.0),
        early_stop_patience=cfg.get("early_stop_patience"),
        early_stop_min_epoch=cfg.get("early_stop_min_epoch", 4),
        restore_best=cfg.get("restore_best", False),
        adversarial_eps=cfg.get("adversarial_eps", 0.0),
        seed=cfg.get("seed", SEED),
    )
    hist = train_classifier(model, data.X_train, data.y_train, data.X_val, data.y_val, tc)
    met = metrics_for(
        model, data.X_train, data.y_train, data.X_val, data.y_val, data.X_test, data.y_test
    )
    BOARD[name] = {
        "train": met.train_acc,
        "val": met.val_acc,
        "test": met.test_acc,
        "gap": met.gap,
    }
    MODELS[name] = model
    print(
        f"{name:18} train={met.train_acc:.3f}  val={met.val_acc:.3f}  "
        f"test={met.test_acc:.3f}  gap={met.gap:.3f}  stopped@{hist.stopped_epoch}"
        f"  ||W||_2={param_l2(model):.1f}"
    )
    plot_hist(hist, name)
    return model, hist, met


## 1. What we're doing

Regularization = any change intended to **cut generalization error, not training error**.

\[
\tilde{J}(\theta; X, y) = J(\theta; X, y) + \alpha\,\Omega(\theta)
\]

In deep learning the winning move is almost never "use a smaller net." It's a **large net + the right regularizer** (bias–variance: buy a little bias, dump a lot of variance).

Penalize **weights, not biases** (ch. 7.1). A bias is one number per unit.

## 2. \(L_2\) / weight decay — Fig 7.1

Quadratic \(J\) around the unregularized minimizer \(w^*\), Hessian \(H\):

\[
\tilde{w} = (H + \alpha I)^{-1} H w^* = Q\,(\Lambda+\alpha I)^{-1}\Lambda\,Q^\top w^*
\]

Component \(i\) is rescaled by \(\lambda_i/(\lambda_i+\alpha)\). Small \(\lambda\) (poorly determined direction) gets smashed toward 0.

**Knob:** `ALPHA`. Re-run. Watch \(\tilde{w}\) slide left along \(w_1\).

In [ ]:
ALPHA = 0.55  # try 0.05, 0.55, 2.0

w_star = np.array([1.8, 0.7])
H = np.diag([0.15, 2.4])  # λ1 small → w1 poorly determined
w_t = l2_shrink_eigen(w_star, H, ALPHA)
print("w*     ", w_star)
print("w_tilde", w_t)
print("shrink ", w_t / w_star, "  (λ/(λ+α) per axis)")

w1 = np.linspace(-0.6, 2.4, 240)
w2 = np.linspace(-1.2, 1.8, 240)
W1, W2 = np.meshgrid(w1, w2)
delta = np.stack([W1 - w_star[0], W2 - w_star[1]], axis=-1)
J = 0.5 * np.einsum("...i,ij,...j->...", delta, H, delta)
R = 0.5 * (W1**2 + W2**2)

fig, ax = plt.subplots(figsize=(5.6, 5.0), layout="constrained")
ax.contour(W1, W2, J, levels=8, colors="#1f77b4")
ax.contour(W1, W2, R, levels=8, colors="#d62728", linestyles="--")
ax.plot(*w_star, "o", color="#1f77b4", ms=8, label=r"$w^*$")
ax.plot(*w_t, "s", color="#d62728", ms=8, label=r"$\tilde{w}$")
ax.axhline(0, color="#ccc", lw=0.6)
ax.axvline(0, color="#ccc", lw=0.6)
ax.set_xlabel(r"$w_1$ poorly determined")
ax.set_ylabel(r"$w_2$ well determined")
ax.set_aspect("equal")
ax.legend(frameon=False)
ax.set_title(rf"$L_2$ geometry, $\alpha={ALPHA}$")
plt.show()


SGD view (eq 7.5): \(w \leftarrow (1-\epsilon\alpha)w - \epsilon\nabla J\). Multiplicative shrink, then the data step.

MAP view: Gaussian prior on \(w\).

## 3. \(L_1\) vs \(L_2\) in 1-D

Diagonal Hessian (eq 7.23):

\[
\tilde{w}_i = \mathrm{sign}(w_i^*)\max\bigl(|w_i^*| - \alpha/H_{ii},\, 0\bigr)
\]

\(L_2\) never hits exact 0 if \(w_i^*\neq 0\). \(L_1\) does. That's feature selection / Laplace prior.

**Knob:** `ALPHA`.

In [ ]:
ALPHA = 1.0  # try 0.2, 1.0, 2.5

w_star = np.linspace(-3, 3, 400)
H = np.ones_like(w_star)
fig, ax = plt.subplots()
ax.plot(w_star, w_star, color="#bbb", label=r"$w^*$")
ax.plot(w_star, l2_shrink_diag(w_star, H, ALPHA), label=r"$L_2$  $\lambda/(\lambda+\alpha)$")
ax.plot(w_star, l1_soft_threshold(w_star, H, ALPHA), label=r"$L_1$  soft-threshold")
ax.axhline(0, color="#aaa", lw=0.6)
ax.axvline(0, color="#aaa", lw=0.6)
ax.set_xlabel(r"$w^*$")
ax.set_ylabel(r"$\tilde{w}$")
ax.set_title(rf"shrinkage, $\alpha={ALPHA}$")
ax.legend(frameon=False)
plt.show()


## 4. Penalties as constraints (ch. 7.2)

\(\tilde{J} = J + \alpha\Omega\) is the Lagrangian for \(\min J\) s.t. \(\Omega(\theta)\le k\). \(L_2\) feasible set is a disk (optimum usually off-axis). \(L_1\) is a diamond (optimum often a **vertex** → zeros).

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.8), layout="constrained")
axes[0].plot(np.cos(theta), np.sin(theta), color="#1f77b4", lw=2)
axes[0].plot([0.72], [0.69], "o", color="#d62728")
axes[0].set_title(r"$L_2$ ball $\|w\|_2 \leq k$")
diamond = np.array([[1, 0], [0, 1], [-1, 0], [0, -1], [1, 0]], float)
axes[1].plot(diamond[:, 0], diamond[:, 1], color="#ff7f0e", lw=2)
axes[1].plot([0.0], [1.0], "o", color="#d62728")
axes[1].set_title(r"$L_1$ diamond $\|w\|_1 \leq k$ (hits axis)")
w1 = np.linspace(-1.4, 1.4, 200)
W1, W2 = np.meshgrid(w1, w1)
J = 0.5 * ((W1 - 1.15) ** 2 / 0.7 + (W2 - 1.05) ** 2 / 0.55)
for ax in axes:
    ax.contour(W1, W2, J, levels=6, colors="#888", linewidths=0.8)
    ax.axhline(0, color="#ccc", lw=0.6)
    ax.axvline(0, color="#ccc", lw=0.6)
    ax.set_aspect("equal")
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
plt.show()


## 5. Linear: diabetes (OLS / ridge / lasso)

10 real features, small \(n\). Lasso should zero a couple of coefficients (`s1`/`s4` in our earlier run) and keep `bmi` + `s5`.

**Knobs:** `RIDGE_ALPHA`, `LASSO_ALPHA`.

In [ ]:
RIDGE_ALPHA = 2.0
LASSO_ALPHA = 0.8

d = load_diabetes_split()
ols = LinearRegression().fit(d.X_train, d.y_train)
ridge = Ridge(alpha=RIDGE_ALPHA).fit(d.X_train, d.y_train)
lasso = Lasso(alpha=LASSO_ALPHA, max_iter=20000).fit(d.X_train, d.y_train)

def mse(m):
    return float(np.mean((m.predict(d.X_test) - d.y_test) ** 2))

for name, m in [("OLS", ols), ("Ridge", ridge), ("Lasso", lasso)]:
    z = int(np.sum(np.abs(m.coef_) < 1e-6))
    print(f"{name:6}  test MSE={mse(m):8.1f}  zeros={z}/{m.coef_.size}")

fig, axes = plt.subplots(3, 1, figsize=(8.5, 6.2), sharex=True, layout="constrained")
for ax, name, w, c in zip(
    axes,
    ["OLS", f"Ridge α={RIDGE_ALPHA}", f"Lasso α={LASSO_ALPHA}"],
    [ols.coef_, ridge.coef_, lasso.coef_],
    ["#4c4c4c", "#1f77b4", "#ff7f0e"],
):
    ax.stem(range(len(w)), w, linefmt=c, markerfmt="o", basefmt="k-")
    ax.set_ylabel(name)
axes[-1].set_xticks(range(len(d.feature_names)))
axes[-1].set_xticklabels(d.feature_names, rotation=40, ha="right")
plt.show()


Closed form check: unregularized least squares with a bias column should match sklearn OLS (cosine ~ 1).

In [ ]:
Xb = np.c_[d.X_train, np.ones(len(d.X_train))]
closed = ridge_normal_equation(Xb, d.y_train, alpha=0.0)
align = float(
    np.dot(closed[:-1], ols.coef_)
    / (np.linalg.norm(closed[:-1]) * np.linalg.norm(ols.coef_) + 1e-12)
)
print("OLS vs closed-form cosine:", round(align, 6))


## 6. Logistic \(L_1\) on breast_cancer

30 real features, tiny train split so unregularized can interpolate. L1 should keep a handful (often `worst concave points`).

sklearn 1.8+: leave `penalty` alone; `l1_ratio=0` is \(L_2\), `l1_ratio=1` is \(L_1\), `C=np.inf` is none.

**Knob:** `C_L1` / `C_L2` — sklearn's *inverse* regularization. Smaller C = more penalty.

In [ ]:
C_L1 = 0.8   # try 0.2 (very sparse) vs 5.0 (almost dense)
C_L2 = 0.5

c = load_cancer_split()
models = {
    "none": LogisticRegression(C=np.inf, max_iter=4000, random_state=0),
    "L2": LogisticRegression(C=C_L2, l1_ratio=0.0, max_iter=4000, random_state=0),
    "L1": LogisticRegression(C=C_L1, l1_ratio=1.0, solver="saga", max_iter=8000, random_state=0),
}
for name, m in models.items():
    m.fit(c.X_train, c.y_train)
    z = int(np.sum(np.abs(m.coef_) < 1e-4))
    print(
        f"{name:5}  train={m.score(c.X_train, c.y_train):.3f}  "
        f"test={m.score(c.X_test, c.y_test):.3f}  zeros={z}/{m.coef_.size}"
    )

w = models["L1"].coef_.ravel()
order = np.argsort(-np.abs(w))
print("\nL1 |w| ranking (nonzero):")
for i in order:
    if abs(w[i]) < 1e-4:
        continue
    print(f"  {c.feature_names[i]:24}  {w[i]:+.3f}")

fig, ax = plt.subplots(figsize=(9, 3.4), layout="constrained")
ax.stem(range(len(w)), w, linefmt="#ff7f0e", markerfmt="o", basefmt="k-")
ax.set_xticks(range(len(c.feature_names)))
ax.set_xticklabels(c.feature_names, rotation=90, fontsize=7)
ax.set_title("logreg L1 weights")
plt.show()


## 7. Fashion-MNIST

28×28 greyscale clothes, 10 classes. First run downloads Zalando's files into `data_cache/` (~30MB).

In [ ]:
DATA = load_fashion_split(n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST, seed=SEED)
print("train", DATA.X_train.shape, "val", DATA.X_val.shape, "test", DATA.X_test.shape)
print("classes", DATA.n_classes, "features", DATA.n_features)

n, side = 40, 28
fig, axes = plt.subplots(4, 10, figsize=(11, 4.4), layout="constrained")
for ax, img, lab in zip(axes.ravel(), DATA.X_train[:n], DATA.y_train[:n]):
    ax.imshow(img.reshape(side, side), cmap="gray")
    ax.set_title(FASHION_LABELS[int(lab)], fontsize=7)
    ax.axis("off")
fig.suptitle("Fashion-MNIST train subset")
plt.show()


## 8. Unregularized MLP — the thing we regularize

Fat 256-256 net, small \(n\). This is the **overfit baseline**. Later cells compare against it via `BOARD` / `show_board()`.

In [ ]:
model_none, hist_none, met_none = fit_mlp("none", restore_best=False, seed=0)


## 9. \(L_2\) weight decay

`AdamW` + `weight_decay` on `ndim>1` tensors only (biases skipped).

**Knob:** `WD`. Too small ≈ none. Too big ≈ underfit. `||W||_2` should drop vs `none`.

In [ ]:
WD = 0.04  # try 1e-3, 0.04, 0.2
_ = fit_mlp("l2", weight_decay=WD, seed=1)
if "none" in MODELS:
    print(f"||W||_2 ratio l2/none = {param_l2(MODELS['l2']) / param_l2(MODELS['none']):.3f}")
show_board()


## 10. \(L_1\) on weights

Added to the loss: \(\alpha\sum |W|\). Should shrink the train–test gap more aggressively than \(L_2\), sometimes at the cost of test acc.

**Knob:** `L1`.

In [ ]:
L1 = 8e-4  # try 1e-4, 8e-4, 3e-3
_ = fit_mlp("l1", l1=L1, seed=2)
print("weight L1", float(weight_l1(MODELS["l1"])))
show_board()


## 11. Dropout (ch. 7.12)

Train-time Bernoulli mask on hidden units. PyTorch's `Dropout` is **inverted** (rescale at train, identity at eval).

Interpretation: noise on hidden units / cheap ensemble of \(2^n\) subnets / breaks co-adaptation.

**Knob:** `DROPOUT`.

In [ ]:
DROPOUT = 0.5  # try 0.2, 0.5, 0.8
_ = fit_mlp("dropout", dropout=DROPOUT, seed=3)
show_board()


## 12. Input noise (ch. 7.5)

Gaussian noise on \(x\) at train time. Infinitesimal noise ≈ weight decay (Bishop); finite noise is strictly more powerful.

**Knob:** `NOISE` (pixel scale is \([0,1]\)).

In [ ]:
NOISE = 0.15  # try 0.05, 0.15, 0.4
_ = fit_mlp("input_noise", input_noise=NOISE, seed=4)
show_board()


## 13. Label smoothing (ch. 7.5.1)

Replace one-hot \(y\) with \((1-\epsilon)y + \epsilon/K\). Stops the net slamming softmax logits to \(\pm\infty\).

**Knob:** `EPS`.

In [ ]:
EPS = 0.1  # try 0.05, 0.1, 0.3
_ = fit_mlp("label_smooth", label_smoothing=EPS, seed=6)
show_board()


## 14. Early stopping (ch. 7.8)

Most-used DL regularizer. Track val, keep the best snapshot, stop after `patience` non-improvements.

Under quadratic \(J\) + GD, early stopping \(\equiv L_2\) (Bishop / Sjöberg–Ljung). Number of steps \(\leftrightarrow 1/\alpha\).

We restore the **best val-acc** snapshot (`restore_best=True`). Dotted line on the loss plot is `best_epoch`. Compare to `none`.

**Knobs:** `PATIENCE`, `MIN_EPOCH` (patience does not count until this epoch — otherwise a lucky epoch-1 val acc kills the run). This cell also uses a few extra epochs so stopping can actually fire.

In [ ]:
PATIENCE = 4   # try 2, 4, 10
MIN_EPOCH = 4  # try 0 (stops too early) vs 8
hist_es = fit_mlp(
    "early_stop",
    epochs=max(EPOCHS, 16),
    early_stop_patience=PATIENCE,
    early_stop_min_epoch=MIN_EPOCH,
    restore_best=True,
    seed=5,
)[1]
print("best_epoch", hist_es.best_epoch, "stopped_epoch", hist_es.stopped_epoch)
show_board()


Stack dropout + L2 if you want. Skip if you're bored.

In [ ]:
_ = fit_mlp("l2+dropout", dropout=0.4, weight_decay=0.02, seed=7)
show_board()


## 15. Parameter sharing — CNN vs MLP (ch. 7.9)

Same 3×3 kernel at every location. Translation prior + far fewer unique params.

This cell trains a TinyCNN on the same Fashion split. Compare test acc and `n_params` to the MLP.

In [ ]:
IMAGES = load_fashion_split(
    n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST, seed=SEED, as_images=True
)
cnn = TinyCNN(img_size=28)
print("CNN params", n_params(cnn), "  MLP params", n_params(MLP(784, 10, hidden=HIDDEN)))
hist_cnn = train_classifier(
    cnn,
    IMAGES.X_train, IMAGES.y_train, IMAGES.X_val, IMAGES.y_val,
    TrainConfig(epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw", seed=11),
)
met_cnn = metrics_for(
    cnn, IMAGES.X_train, IMAGES.y_train, IMAGES.X_val, IMAGES.y_val, IMAGES.X_test, IMAGES.y_test
)
BOARD["cnn"] = {"train": met_cnn.train_acc, "val": met_cnn.val_acc, "test": met_cnn.test_acc, "gap": met_cnn.gap}
MODELS["cnn"] = cnn
print(f"cnn  train={met_cnn.train_acc:.3f}  test={met_cnn.test_acc:.3f}  gap={met_cnn.gap:.3f}")
plot_hist(hist_cnn, "cnn")
show_board()


## 16. Dataset augmentation (ch. 7.4)

Manufacture \((x,y)\) by transforming \(x\) without changing the label. Here: \(\pm\) pixel rolls. Do **not** flip — 6/9 (and shirts vs not) is how you silently poison the label.

**Knob:** `MAX_SHIFT`, `N_COPIES`.

In [ ]:
MAX_SHIFT = 2     # try 1, 2, 4
N_COPIES = 2      # extra shifted copies of the train set

aug_X = [IMAGES.X_train]
aug_y = [IMAGES.y_train]
for i in range(N_COPIES):
    aug_X.append(shift_images(IMAGES.X_train, max_shift=MAX_SHIFT, seed=20 + i))
    aug_y.append(IMAGES.y_train)
X_aug, y_aug = np.concatenate(aug_X), np.concatenate(aug_y)
print("train size", len(IMAGES.y_train), "→", len(y_aug))

cnn_aug = TinyCNN(img_size=28)
hist_aug = train_classifier(
    cnn_aug, X_aug, y_aug, IMAGES.X_val, IMAGES.y_val,
    TrainConfig(epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw", seed=12),
)
met_aug = metrics_for(
    cnn_aug, IMAGES.X_train, IMAGES.y_train, IMAGES.X_val, IMAGES.y_val, IMAGES.X_test, IMAGES.y_test
)
BOARD["cnn+aug"] = {"train": met_aug.train_acc, "val": met_aug.val_acc, "test": met_aug.test_acc, "gap": met_aug.gap}
MODELS["cnn+aug"] = cnn_aug
print(f"cnn+aug  train={met_aug.train_acc:.3f}  test={met_aug.test_acc:.3f}  gap={met_aug.gap:.3f}")
plot_hist(hist_aug, "cnn+aug")
show_board()


## 17. Sparse *representations* (ch. 7.10)

Two different sparsities:

| | what is zero | mechanism |
|---|---|---|
| sparse **parameters** | weights | \(L_1\) on \(W\) (cell 10) |
| sparse **activations** | hidden \(h\) | \(L_1\) on \(h\) (this cell) |

A dense \(W\) can still map \(x\) to a sparse \(h\). Tanh without a penalty saturates at \(\pm 1\); L1 on \(h\) piles mass at 0.

**Knob:** `ACT_L1`.

In [ ]:
ACT_L1 = 0.15  # try 0, 0.05, 0.15, 0.4

def hidden_stats(act_l1: float, name: str):
    torch.manual_seed(30)
    model = MLP(DATA.n_features, DATA.n_classes, hidden=(64, 64), activation="tanh")
    train_classifier(
        model, DATA.X_train, DATA.y_train, DATA.X_val, DATA.y_val,
        TrainConfig(
            epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw",
            activation_l1=act_l1, seed=30,
        ),
    )
    model.eval()
    with torch.no_grad():
        _, h = model(torch.from_numpy(DATA.X_test[:800]), return_hidden=True)
    h = h.numpy()
    print(f"{name:18} mean|h|={np.mean(np.abs(h)):.3f}  frac≈0={np.mean(np.abs(h)<1e-2):.3f}")
    return h

h0 = hidden_stats(0.0, "tanh")
h1 = hidden_stats(ACT_L1, "tanh + L1 on h")

fig, ax = plt.subplots()
ax.hist(h0.ravel(), bins=60, density=True, alpha=0.55, label="no act. penalty")
ax.hist(h1.ravel(), bins=60, density=True, alpha=0.55, label=f"L1={ACT_L1} on h")
ax.set_xlabel("hidden activation")
ax.set_ylabel("density")
ax.legend(frameon=False)
plt.show()


## 18. Bagging (ch. 7.11)

Train \(k\) models on bootstrap resamples, average the softmaxes. Different from dropout: here the ensemble is explicit and the members see different datasets.

The `none` MLP is already on the board — this cell trains `N_BAG` extra nets. Slow-ish; drop `N_BAG` to 2 if you're impatient.

**Knob:** `N_BAG`.

In [ ]:
N_BAG = 3  # try 2, 3, 5

members = []
for i in range(N_BAG):
    idx = bootstrap_indices(len(DATA.X_train), seed=20 + i)
    m = MLP(DATA.n_features, DATA.n_classes, hidden=HIDDEN)
    train_classifier(
        m,
        DATA.X_train[idx], DATA.y_train[idx], DATA.X_val, DATA.y_val,
        TrainConfig(epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw", seed=20 + i),
    )
    members.append(m)
    print(f"  member {i} test={metrics_for(m, DATA.X_train, DATA.y_train, DATA.X_val, DATA.y_val, DATA.X_test, DATA.y_test).test_acc:.3f}")

bag_tr = bagged_proba(members, DATA.X_train).argmax(1)
bag_va = bagged_proba(members, DATA.X_val).argmax(1)
bag_te = bagged_proba(members, DATA.X_test).argmax(1)
tr = float(np.mean(bag_tr == DATA.y_train))
va = float(np.mean(bag_va == DATA.y_val))
te = float(np.mean(bag_te == DATA.y_test))
BOARD[f"bag x{N_BAG}"] = {"train": tr, "val": va, "test": te, "gap": tr - te}
print(f"bag x{N_BAG: <13} train={tr:.3f}  val={va:.3f}  test={te:.3f}  gap={tr-te:.3f}")
show_board()


## 19. Adversarial training / FGSM (ch. 7.13)

\[
x_{\mathrm{adv}} = x + \varepsilon\,\mathrm{sign}(\nabla_x J)
\]

Worst-case perturbation, not isotropic noise. Train on a 50/50 mix of clean + FGSM.

**Knob:** `ADV_EPS`.

In [ ]:
ADV_EPS = 0.12  # try 0.03, 0.12, 0.25

def acc(model, X, y, batch=128):
    model.eval()
    n = 0
    correct = 0
    xt = torch.from_numpy(np.ascontiguousarray(X))
    yt = torch.from_numpy(np.ascontiguousarray(y)).long()
    for i in range(0, len(X), batch):
        pred = model(xt[i:i + batch]).argmax(1)
        correct += int((pred == yt[i:i + batch]).sum())
        n += pred.numel()
    return correct / n


def fgsm_acc(model, X, y, eps, batch=128):
    model.eval()
    n = 0
    correct = 0
    xt = torch.from_numpy(np.ascontiguousarray(X))
    yt = torch.from_numpy(np.ascontiguousarray(y)).long()
    for i in range(0, len(X), batch):
        xb, yb = xt[i:i + batch], yt[i:i + batch]
        adv = fgsm(model, xb, yb, eps)
        pred = model(adv).argmax(1)
        correct += int((pred == yb).sum())
        n += yb.numel()
    return correct / n

rows = []
for name, adv in [("clean train", 0.0), ("FGSM train", ADV_EPS)]:
    m = TinyCNN(img_size=28)
    train_classifier(
        m, IMAGES.X_train, IMAGES.y_train, IMAGES.X_val, IMAGES.y_val,
        TrainConfig(
            epochs=EPOCHS, lr=LR, batch_size=64, optimizer="adamw",
            adversarial_eps=adv, seed=40,
        ),
    )
    rows.append({
        "name": name,
        "clean": acc(m, IMAGES.X_test, IMAGES.y_test),
        "fgsm": fgsm_acc(m, IMAGES.X_test, IMAGES.y_test, ADV_EPS),
    })
    print(f"{name:12}  clean={rows[-1]['clean']:.3f}  FGSM={rows[-1]['fgsm']:.3f}")

fig, ax = plt.subplots(figsize=(6.4, 3.8))
x = np.arange(len(rows))
ax.bar(x - 0.18, [r["clean"] for r in rows], 0.36, label="clean test")
ax.bar(x + 0.18, [r["fgsm"] for r in rows], 0.36, label="FGSM test", color="#d62728")
ax.set_xticks(x)
ax.set_xticklabels([r["name"] for r in rows])
ax.set_ylim(0, 1.05)
ax.legend(frameon=False)
ax.set_title(rf"FGSM $\varepsilon={ADV_EPS}$")
plt.show()


## 20. Scoreboard

Re-run after any cell. `gap = train − test` on the (noisy-free) Fashion split.

In [ ]:
show_board()

names = list(BOARD)
fig, ax = plt.subplots(figsize=(max(6.5, 0.7 * len(names)), 4.0), layout="constrained")
x = np.arange(len(names))
ax.bar(x - 0.18, [BOARD[k]["train"] for k in names], 0.36, label="train", color="#9ecae1")
ax.bar(x + 0.18, [BOARD[k]["test"] for k in names], 0.36, label="test", color="#2171b5")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right")
ax.set_ylim(0.4, 1.03)
ax.set_ylabel("accuracy")
ax.legend(frameon=False)
plt.show()


## 21. Things to try next

- Crank `EPOCHS` to 25 and `N_TRAIN` to 1500 (setup cell) — closer to `run.py`.
- `none` vs `early_stop`: overlay `hist_none.val_loss` and the early-stop snapshot epoch.
- Replace Fashion with `load_cancer_split` + a tiny MLP — L1 feature selection vs dropout on tabular data.
- Combo: `dropout=0.4, input_noise=0.1, label_smoothing=0.05`.
- FGSM \(\varepsilon\) vs accuracy curve: loop `ADV_EPS` in `{0.03, 0.06, 0.12, 0.25}`.
- Bag vs dropout: same compute budget (3 nets × \(T\) vs 1 net × \(3T\)).

Interview list is in `NOTES.md` (why \(L_2\) never zeros, bias skip, inverted dropout, sharing vs tying, noise vs FGSM).